# Malaria Hybrid Clinical AI Demo

This notebook demonstrates the capstone-ready malaria severity prototype. The deployed system combines a V3 RandomForest model, a FastAPI backend, a Streamlit frontend, a clinical safety layer, and SHAP explainability.

## 1. Project Introduction

The project is a research prototype for safety-aware explainable clinical decision support. It estimates severe malaria risk from demographic and symptom inputs, then applies a transparent clinical safety layer for high-risk indicators. It is not a medical device and must not be used as a final diagnosis.

## 2. Dataset Overview

The project data is stored at `data/Malaria-Data.csv`. The active target used by the V3 trainer is `severe_malaria`.

In [ ]:
import pandas as pd

data_path = "../data/Malaria-Data.csv"
df = pd.read_csv(data_path)
df.head()

## 3. Feature Description

The active feature schema is loaded from `model/features_v3.joblib`. It includes demographic variables and malaria symptom indicators used by the deployed backend.

In [ ]:
import joblib

features = joblib.load("../model/features_v3.joblib")
features

## 4. Model Loading

The production backend loads the V3 RandomForest artifacts below. This notebook uses the same files for demonstration.

In [ ]:
model = joblib.load("../model/model_v3.joblib")
model

## 5. Example Prediction

The example below creates a single patient payload using the deployed feature order and obtains model probability.

In [ ]:
sample = {feature: 0 for feature in features}
sample.update({
    "age": 25,
    "sex": 0,
    "fever": 1,
    "rigor": 1,
    "fatigue": 1,
    "headache": 1,
    "diarrhea": 1,
})

X = pd.DataFrame([[float(sample[f]) for f in features]], columns=features)
prediction = int(model.predict(X)[0])
probability_severe = float(model.predict_proba(X)[0][1])

{
    "model_prediction": prediction,
    "probability_severe": round(probability_severe, 4),
}

## 6. Hybrid Safety Override

The deployed API keeps the model prediction visible but layers a clinical safety assessment on top. Critical indicators such as convulsion, hypoglycemia, prostration, hyperpyrexia, jaundice, or coca-cola urine can trigger safety escalation or safety confirmation depending on whether the model already predicted severe malaria.

## 7. SHAP Explainability

SHAP values explain the model probability layer. The clinical safety layer is reported separately in the API evidence fields and dashboard reasoning text.

In [ ]:
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

if isinstance(shap_values, list):
    severe_shap_values = shap_values[1][0]
else:
    severe_shap_values = shap_values[0, :, 1]

pd.DataFrame({
    "feature": features,
    "shap_contribution": severe_shap_values,
}).sort_values("shap_contribution", key=lambda s: s.abs(), ascending=False).head(10)

## 8. Sample API Call

Run the backend locally with `uvicorn src.app:app --reload`, then call `/predict`.

In [ ]:
import requests

api_payload = sample.copy()
# response = requests.post("http://127.0.0.1:8000/predict", json=api_payload, timeout=15)
# response.json()

## 9. Screenshot Placeholders

- Screenshot 1: Baseline or minimal symptom prediction.
- Screenshot 2: General symptoms only prediction.
- Screenshot 3: Critical indicator safety escalation or confirmation.
- Screenshot 4: SHAP explainability table/chart.

## 10. Conclusion

This notebook supports lecturer demonstration, GitHub portfolio review, and technical interview discussion. The production application remains the FastAPI and Streamlit implementation in `src/app.py` and `src/ui.py`.